# 💳 Credit Card Fraud Detection (Colab Edition)

This notebook trains and compares three models:
1. **Random Forest Classifier**
2. **XGBoost Classifier**
3. **Dense Neural Network (TensorFlow / Keras)**

### Recommended Setup:
- Set Runtime to **T4 GPU** for faster training (`Runtime` -> `Change runtime type` -> `T4 GPU`).

### Step 1: Download or Upload Dataset (`creditcard.csv`)
Run this cell to automatically download the dataset directly into Google Colab.

In [ ]:
# Download creditcard.csv if not present
import os

if not os.path.exists('creditcard.csv'):
    print('Downloading creditcard.csv dataset...')
    !pip install -q kagglehub
    import kagglehub
    path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")
    import shutil
    source_file = os.path.join(path, "creditcard.csv")
    if os.path.exists(source_file):
        shutil.copy(source_file, "creditcard.csv")
        print('Dataset ready: creditcard.csv')
    else:
        print('Files in dataset path:', os.listdir(path))
else:
    print('creditcard.csv is already present!')

### Step 2: Data Exploration & Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

df = pd.read_csv("creditcard.csv")
print("Shape:", df.shape)
df = df.drop_duplicates()
print("Shape after dropping duplicates:", df.shape)
df.head()

In [ ]:
# Check class distribution
count_classes = df['Class'].value_counts().sort_index()
print(count_classes)

sns.barplot(x=['Non-fraud (0)', 'Fraud (1)'], y=count_classes.values, palette=['green', 'red'])
plt.title("Transaction Class Distribution")
plt.xlabel("Class")
plt.ylabel("Frequency")
plt.yscale('log')
plt.show()

In [ ]:
# Feature preparation
# Remove Time and scale Amount
df = df.drop(['Time'], axis=1, errors='ignore')
df['Amount'] = StandardScaler().fit_transform(df['Amount'].values.reshape(-1, 1))

X = df.drop(['Class'], axis=1)
y = df['Class']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training set: {X_train.shape}, Test set: {X_test.shape}")

### Step 3: Model 1 - Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score

rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

print("Training Random Forest Classifier...")
rf_model.fit(X_train, y_train)
print("Random Forest Training Complete.")

y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

print("\n--- Random Forest Classifier Results ---")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1-Score:", f1_score(y_test, y_pred_rf))
print("ROC AUC Score:", roc_auc_score(y_test, y_proba_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

### Step 4: Model 2 - XGBoost Classifier

In [ ]:
import xgboost as xgb

xgb_model = xgb.XGBClassifier(
    objective='binary:logistic',
    n_estimators=100,
    learning_rate=0.1,
    eval_metric='logloss',
    random_state=42,
    n_jobs=-1
)

print("Training XGBoost Classifier...")
xgb_model.fit(X_train, y_train)
print("XGBoost Training Complete.")

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("\n--- XGBoost Classifier Results ---")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1-Score:", f1_score(y_test, y_pred_xgb))
print("ROC AUC Score:", roc_auc_score(y_test, y_proba_xgb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb))

### Step 5: Model 3 - Dense Neural Network (Deep Learning)

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

input_dim = X_train.shape[1]

nn_model = keras.Sequential([
    layers.Input(shape=(input_dim,)),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
    layers.Dense(1, activation='sigmoid')
])

nn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall(), tf.keras.metrics.AUC()]
)

print("Training Dense Neural Network...")
history = nn_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)
print("Neural Network Training Complete.")

y_proba_nn = nn_model.predict(X_test).flatten()
y_pred_nn = (y_proba_nn > 0.5).astype(int)

print("\n--- Dense Neural Network Results ---")
print("Accuracy:", accuracy_score(y_test, y_pred_nn))
print("Precision:", precision_score(y_test, y_pred_nn))
print("Recall:", recall_score(y_test, y_pred_nn))
print("F1-Score:", f1_score(y_test, y_pred_nn))
print("ROC AUC Score:", roc_auc_score(y_test, y_proba_nn))
print("\nClassification Report:\n", classification_report(y_test, y_pred_nn))

### Step 6: Summary & Comparison

In [ ]:
models_summary = pd.DataFrame({
    'Model': ['Random Forest', 'XGBoost', 'Dense Neural Network'],
    'Accuracy': [accuracy_score(y_test, y_pred_rf), accuracy_score(y_test, y_pred_xgb), accuracy_score(y_test, y_pred_nn)],
    'Precision': [precision_score(y_test, y_pred_rf), precision_score(y_test, y_pred_xgb), precision_score(y_test, y_pred_nn)],
    'Recall': [recall_score(y_test, y_pred_rf), recall_score(y_test, y_pred_xgb), recall_score(y_test, y_pred_nn)],
    'F1-Score': [f1_score(y_test, y_pred_rf), f1_score(y_test, y_pred_xgb), f1_score(y_test, y_pred_nn)],
    'ROC AUC': [roc_auc_score(y_test, y_proba_rf), roc_auc_score(y_test, y_proba_xgb), roc_auc_score(y_test, y_proba_nn)]
})

print(models_summary.to_string(index=False))
best_model = models_summary.loc[models_summary['F1-Score'].idxmax()]['Model']
print(f"\n🏆 Best Model by F1-Score: {best_model}")